# ViBioMIR PA5 trên Kaggle

Notebook chạy BM25 + BGE-M3, xếp hạng tài liệu, chọn đoạn và xuất `submission.zip`.
Chế độ mặc định dùng trọng số có sẵn. Có thể bật fine-tune BGE-M3 nếu có dữ liệu huấn luyện gắn nhãn.

1. Chọn **GPU** trong **Session options**. Bật **Internet** để tải thư viện và model, hoặc dùng wheel và model đã gắn vào Input.
2. Dùng **Add Input** để gắn dataset mã nguồn chứa `src/` và `requirements.txt` cùng cấp. Có thể thay bằng URL Git HTTPS nếu repo truy cập được từ Kaggle.
3. Gắn dataset chứa `corpus_shards/*.parquet` với các cột `id`, `content`, `language` (vi/en/zh) và `queries_expanded.json` hoặc `query.csv` với cột `id,query`. Nếu chỉ có dữ liệu crawl, cần chuẩn bị Parquet trước.
4. Chạy các ô theo thứ tự. Giữ `SMOKE_TEST = True` cho lần đầu, sau đó đổi sang `False` để tạo submission đầy đủ.

[Hướng dẫn Kaggle Notebooks](https://www.kaggle.com/docs/notebooks)

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

SOURCE_DIR = None         # VD "/kaggle/input/datasets/owner/rag-code"
REPO_URL = None           # VD "https://github.com/owner/repo.git"
DATA_ROOT = INPUT_ROOT    # Thu hẹp đến một dataset nếu có nhiều bộ dữ liệu
CORPUS_DIR = None         # Tùy chọn, đường dẫn trực tiếp đến corpus_shards
QUERIES_FILE = None       # Tùy chọn, đường dẫn trực tiếp đến queries_expanded.json
QUERY_CSV = None          # Tùy chọn, dùng riêng CSV nếu muốn bỏ qua JSON
STATE_ROOT = None         # Tùy chọn, thư mục chứa state từ lần chạy trước
WHEEL_DIR = None          # Tùy chọn, thư mục wheel nếu chạy offline

MODEL = "BAAI/bge-m3"     # Hoặc thư mục chứa checkpoint BGE-M3
RUN_NAME = "pa5"
SMOKE_TEST = True
SMOKE_QUERIES = 3
SMOKE_DOCS_PER_LANGUAGE = 30
EXPORT_STATE = True
RUN_TIMEOUT_MINUTES = None  # VD 480 để chừa thời gian đóng gói trước khi phiên hết hạn

TRAIN_JSONL = None        # Đặt đường dẫn để bật fine-tune có giám sát
TRAIN_OUTPUT = WORKING / "bge-m3-finetuned"
QRELS_JSON = None         # Tùy chọn, JSON {"query_id": ["doc_id", ...]} cho tập đánh giá

env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUNBUFFERED": "1"}


def find_one(root, name, predicate, required=False):
    candidates = sorted(path for path in root.rglob(name) if predicate(path))
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều lựa chọn cho {name}. Hãy đặt đường dẫn cụ thể: {candidates}")
    if not candidates and required:
        raise FileNotFoundError(f"Không tìm thấy {name} trong {root}")
    return candidates[0] if candidates else None

## Mã nguồn và thư viện

In [ ]:
if SOURCE_DIR:
    project = Path(SOURCE_DIR)
elif REPO_URL:
    project = WORKING / "rag-source"
    if not project.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project)], check=True)
else:
    requirements = find_one(
        INPUT_ROOT, "requirements.txt",
        lambda path: (path.parent / "src" / "__main__.py").is_file(), required=True,
    )
    project = requirements.parent

if not (project / "src" / "__main__.py").is_file():
    raise FileNotFoundError(f"Thiếu src/__main__.py trong {project}")

pip_args = [sys.executable, "-m", "pip", "install"]
if WHEEL_DIR:
    pip_args += ["--no-index", "--find-links", str(WHEEL_DIR)]
subprocess.run(pip_args + ["-r", str(project / "requirements.txt")], check=True)
subprocess.run(
    [sys.executable, "-c",
     "import torch; from FlagEmbedding import BGEM3FlagModel; "
     "assert torch.cuda.is_available(), 'Hãy chọn GPU trong Session options'; "
     "print(torch.__version__, torch.cuda.get_device_name(0))"],
    check=True,
)

## Kiểm tra corpus và truy vấn

In [ ]:
root = Path(DATA_ROOT)
corpus = Path(CORPUS_DIR) if CORPUS_DIR else find_one(root, "corpus_shards", Path.is_dir, required=True)
expanded = Path(QUERIES_FILE) if QUERIES_FILE else None
query_csv = Path(QUERY_CSV) if QUERY_CSV else None
if expanded is None and query_csv is None:
    expanded = find_one(root, "queries_expanded.json", Path.is_file)
    if expanded is None:
        query_csv = find_one(root, "query.csv", Path.is_file, required=True)

paths = [path for path in (expanded, query_csv) if path is not None]
for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)

import pyarrow as pa
import pyarrow.parquet as pq

columns = ["id", "content", "language"]
languages = ("vi", "en", "zh")
shards = sorted(corpus.glob("*.parquet"))
if not shards:
    raise FileNotFoundError(f"Không có shard Parquet trong {corpus}")
total_docs = 0
for shard in shards:
    parquet = pq.ParquetFile(shard)
    missing = set(columns) - set(parquet.schema_arrow.names)
    if missing:
        raise ValueError(f"{shard.name} thiếu cột {sorted(missing)}")
    total_docs += parquet.metadata.num_rows
if not total_docs:
    raise ValueError("Corpus rỗng.")

subprocess.run(
    [sys.executable, "-c",
     "from pathlib import Path; from src.data import load_queries; import sys; "
     "load_queries(*(Path(arg) if arg else None for arg in sys.argv[1:]))",
     str(expanded or ""), str(query_csv or "")],
    cwd=project, env=env, check=True,
)
print(f"Corpus có {total_docs:,} tài liệu trong {len(shards)} shard.")
print("Dung lượng trống:", round(shutil.disk_usage(WORKING).free / 2**30, 1), "GiB")

## Fine-tune BGE-M3 tùy chọn

Để trống `TRAIN_JSONL` để dùng trực tiếp model trong `MODEL`. Nếu bật fine-tune, dùng file JSONL, mỗi dòng có `query` là chuỗi cùng `pos` và `neg` là danh sách văn bản. Các mục `pos` là đoạn liên quan, `neg` là đoạn không liên quan. Dùng tập huấn luyện riêng với tập đánh giá. Ô này chỉ huấn luyện dense encoder và dùng cấu hình nhỏ cho một GPU.

[Định dạng dữ liệu FlagEmbedding](https://bge-model.com/tutorial/7_Finetuning/7.1.2.html) · [Lệnh fine-tune BGE-M3](https://github.com/FlagOpen/FlagEmbedding/blob/master/examples/finetune/embedder/encoder_only/m3.sh)

In [ ]:
import json

selected_model = str(MODEL)
if TRAIN_JSONL is not None:
    train_file = Path(TRAIN_JSONL).resolve(strict=True)
    examples = 0
    with train_file.open(encoding="utf-8") as reader:
        for line_number, line in enumerate(reader, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            valid = (
                isinstance(row, dict)
                and isinstance(row.get("query"), str)
                and bool(row["query"].strip())
                and all(
                    isinstance(row.get(key), list)
                    and bool(row[key])
                    and all(isinstance(item, str) and item.strip() for item in row[key])
                    for key in ("pos", "neg")
                )
            )
            if not valid:
                raise ValueError(f"Dòng {line_number} thiếu query, pos hoặc neg hợp lệ.")
            examples += 1
    if not examples:
        raise ValueError("File huấn luyện rỗng.")
    if TRAIN_OUTPUT.exists():
        raise FileExistsError(f"Đổi TRAIN_OUTPUT hoặc đặt MODEL trỏ đến {TRAIN_OUTPUT} để dùng lại model.")

    command = [
        sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=1",
        "-m", "FlagEmbedding.finetune.embedder.encoder_only.m3",
        "--model_name_or_path", selected_model, "--train_data", str(train_file),
        "--output_dir", str(TRAIN_OUTPUT), "--cache_path", str(Path("/kaggle/temp") / "pa5-train-cache"),
        "--do_train", "--fp16", "--gradient_checkpointing", "--optim", "adamw_torch",
        "--train_group_size", "2", "--query_max_len", "128", "--passage_max_len", "256",
        "--per_device_train_batch_size", "1", "--gradient_accumulation_steps", "8",
        "--learning_rate", "1e-5", "--num_train_epochs", "1",
        "--normalize_embeddings", "True", "--unified_finetuning", "False",
        "--logging_steps", "10", "--report_to", "none", "--save_strategy", "no",
    ]
    subprocess.run(command, cwd=project, env=env, check=True)
    if not (TRAIN_OUTPUT / "config.json").is_file():
        raise FileNotFoundError(f"Thiếu config.json trong model đã train: {TRAIN_OUTPUT}")
    selected_model = str(TRAIN_OUTPUT)
print("Model dùng để truy xuất:", selected_model)

## Chạy truy xuất và xuất submission

Chế độ chạy thử lấy một mẫu nhỏ từ corpus và tối đa `SMOKE_QUERIES` truy vấn. Kết quả chạy thử dùng để kiểm tra pipeline. Đặt `SMOKE_TEST = False` để xử lý đầy đủ.

Để tiếp tục phiên Kaggle mới, gắn `pa5_state.zip` từ phiên cũ, đặt `STATE_ROOT` là thư mục chứa ZIP, giữ nguyên corpus, truy vấn và model. Checkpoint tạo trước khi có kiểm tra chữ ký đầu vào cần được xây lại trong thư mục mới. Khi đặt giới hạn thời gian, chừa thời gian để nén state trước khi phiên Kaggle kết thúc.

In [ ]:
artifact_dir = WORKING / (f"{RUN_NAME}_smoke" if SMOKE_TEST else RUN_NAME)
work_dir = Path("/kaggle/temp") / artifact_dir.name
artifact_dir.mkdir(parents=True, exist_ok=True)
work_dir.mkdir(parents=True, exist_ok=True)
run_corpus = corpus

if SMOKE_TEST:
    samples = {lang: [] for lang in languages}
    for shard in shards:
        for batch in pq.ParquetFile(shard).iter_batches(columns=columns, batch_size=1024):
            for row in batch.to_pylist():
                bucket = samples.get(row["language"])
                if bucket is not None and len(bucket) < SMOKE_DOCS_PER_LANGUAGE:
                    bucket.append(row)
            break
        if all(len(rows) >= SMOKE_DOCS_PER_LANGUAGE for rows in samples.values()):
            break

    sample_rows = [row for rows in samples.values() for row in rows]
    if not sample_rows:
        raise ValueError("Không tìm thấy tài liệu vi/en/zh để chạy thử.")
    print("Tài liệu chạy thử:", {lang: len(rows) for lang, rows in samples.items()})
    run_corpus = work_dir / "smoke_corpus"
    run_corpus.mkdir(exist_ok=True)
    pq.write_table(pa.Table.from_pylist(sample_rows), run_corpus / "sample.parquet")

command = [
    sys.executable, "-m", "src", "--corpus-dir", str(run_corpus),
    "--work-dir", str(work_dir), "--artifact-dir", str(artifact_dir),
    "--state-root", str(Path(STATE_ROOT) if STATE_ROOT and not SMOKE_TEST else work_dir),
    "--model", selected_model,
]
if expanded:
    command += ["--queries-file", str(expanded)]
if query_csv:
    command += ["--query-csv", str(query_csv)]
if SMOKE_TEST:
    command += ["--limit-queries", str(SMOKE_QUERIES)]
if SMOKE_TEST or not EXPORT_STATE:
    command += ["--skip-state-export"]

try:
    subprocess.run(
        command, cwd=project, env=env, check=True,
        timeout=RUN_TIMEOUT_MINUTES * 60 if RUN_TIMEOUT_MINUTES else None,
    )
except (subprocess.TimeoutExpired, subprocess.CalledProcessError):
    if EXPORT_STATE and not SMOKE_TEST and (work_dir / "checkpoint.txt").is_file():
        partial_dir = artifact_dir / "interrupted"
        subprocess.run(
            [sys.executable, "-c",
             "from pathlib import Path; from src.export import export_state; import sys; "
             "export_state(Path(sys.argv[1]), Path(sys.argv[2]))",
             str(work_dir), str(partial_dir)],
            cwd=project, env=env, check=True,
        )
        print("Checkpoint của phiên gián đoạn:", partial_dir / "pa5_state.zip")
        print("Giữ lại cả state đầu vào nếu phiên này đã tiếp tục từ state cũ.")
    raise

## Kiểm tra kết quả và đo chất lượng khi có nhãn

In [ ]:
from zipfile import ZipFile
from statistics import mean

submission_zip = artifact_dir / "submission.zip"
with ZipFile(submission_zip) as archive:
    with archive.open("submission.json") as file:
        submission = json.load(file)

query_count = int(subprocess.check_output(
    [sys.executable, "-c",
     "from pathlib import Path; from src.data import load_queries; import sys; "
     "a, b, limit = sys.argv[1:]; "
     "q = load_queries(Path(a) if a else None, Path(b) if b else None, int(limit) or None); "
     "print(len(q))",
     str(expanded or ""), str(query_csv or ""), str(SMOKE_QUERIES if SMOKE_TEST else 0)],
    cwd=project, env=env, text=True,
).splitlines()[-1])
results = {str(row["id"]): row for row in submission}
if len(submission) != query_count or len(results) != query_count:
    raise ValueError(f"Cần {query_count} truy vấn duy nhất. Nhận {len(submission)} dòng và {len(results)} ID.")

for field in ("relevant_docs", "relevant_chunks"):
    counts = [len(row[field]) for row in submission]
    print(field, "| trung bình:", round(mean(counts), 1), "| truy vấn rỗng:", counts.count(0))

if QRELS_JSON and not SMOKE_TEST:
    qrels = json.loads(Path(QRELS_JSON).read_text(encoding="utf-8"))
    recall, reciprocal_rank = [], []
    for query_id, doc_ids in qrels.items():
        gold = {str(doc_id) for doc_id in doc_ids}
        if not gold:
            continue
        top = results.get(str(query_id), {}).get("relevant_docs", [])[:10]
        recall.append(len(gold.intersection(top)) / len(gold))
        reciprocal_rank.append(next((1 / i for i, doc in enumerate(top, 1) if doc in gold), 0))
    if recall:
        print(f"Recall@10 = {mean(recall):.4f} | MRR@10 = {mean(reciprocal_rank):.4f} | n = {len(recall)}")

for path in sorted(artifact_dir.glob("*.zip")):
    print(path, f"({path.stat().st_size / 2**20:.1f} MiB)")